# Predict Personal Images with Model v6
Notebook này load lại trọng số của model v6 (Qwen2-VL-2B-Instruct + LoRA + MultiScale Head) trong project hiện tại để dự đoán tính chất Hateful trên các ảnh cá nhân không có nhãn.

In [ ]:
import os
import json
from pathlib import Path
import torch
import torch.nn as nn
import numpy as np
from PIL import Image, ImageFile
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel

ImageFile.LOAD_TRUNCATED_IMAGES = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# Khai báo các đường dẫn đến model v6 đã train
MODEL_ID = 'Qwen/Qwen2-VL-2B-Instruct'

if os.path.exists('/kaggle'):
    print('Running on Kaggle')
    # Lưu ý: upload folder adapter, file multiscale_head.pt và folder personal_images vào chung 1 dataset tên là 'my-model-v6'
    ADAPTER_DIR = Path('/kaggle/input/my-model-v6/adapter')
    CLS_HEAD_PT = Path('/kaggle/input/my-model-v6/multiscale_head.pt')
    PERSONAL_IMAGES_DIR = Path('/kaggle/input/my-model-v6/personal_images')
else:
    print('Running locally')
    TRAINED_MODEL_DIR = Path('Merged Hateful Meme/train/trained_model_v6_multiscale')
    ADAPTER_DIR = TRAINED_MODEL_DIR / 'adapter'
    CLS_HEAD_PT = TRAINED_MODEL_DIR / 'multiscale_head.pt'
    PERSONAL_IMAGES_DIR = Path('personal_images')
    PERSONAL_IMAGES_DIR.mkdir(exist_ok=True)

print(f'Thư mục ảnh hiện tại: {PERSONAL_IMAGES_DIR.absolute()}')

In [ ]:
class MultiScaleClassificationHead(nn.Module):
    def __init__(self, hidden_size: int = 2048, n_scales: int = 3, dropout: float = 0.5):
        super().__init__()
        in_dim = hidden_size * n_scales
        self.input_norm = nn.LayerNorm(in_dim)
        self.fc1   = nn.Linear(in_dim, 512)
        self.norm1 = nn.LayerNorm(512)
        self.act   = nn.GELU()
        self.drop1 = nn.Dropout(dropout)
        self.fc2   = nn.Linear(512, 128)
        self.norm2 = nn.LayerNorm(128)
        self.drop2 = nn.Dropout(dropout)
        self.out   = nn.Linear(128, 1)

    def forward(self, pooled_concat: torch.Tensor) -> torch.Tensor:
        x = self.input_norm(pooled_concat)
        h = self.drop1(self.act(self.norm1(self.fc1(x))))
        h = self.drop2(self.act(self.norm2(self.fc2(h))))
        return self.out(h).squeeze(-1)

In [ ]:
print('Loading processor...')
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

print('Loading base model in 4-bit...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    attn_implementation='sdpa',
    trust_remote_code=True,
)
base_model.config.use_cache = False

print('Loading LoRA adapter...')
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

print('Loading MultiScale Head...')
HIDDEN_SIZE = base_model.config.text_config.hidden_size
cls_head = MultiScaleClassificationHead(hidden_size=HIDDEN_SIZE, n_scales=3)
cls_head.load_state_dict(torch.load(str(CLS_HEAD_PT), map_location=DEVICE, weights_only=False))
cls_head.to(DEVICE)
cls_head.eval()

print('Model v6 loaded successfully!')

In [ ]:
def forward_multiscale(batch):
    outputs = model(
        input_ids=batch['input_ids'],
        attention_mask=batch['attention_mask'],
        pixel_values=batch['pixel_values'],
        image_grid_thw=batch['image_grid_thw'],
        mm_token_type_ids=batch.get('mm_token_type_ids'),
        output_hidden_states=True,
        return_dict=True,
    )
    
    seq_lengths = batch['attention_mask'].sum(dim=1) - 1
    batch_idx = torch.arange(seq_lengths.size(0), device=DEVICE)
    
    pooled_per_scale = []
    for idx in [-1, -6, -12]:
        h = outputs.hidden_states[idx]
        pooled = h[batch_idx, seq_lengths].float()
        pooled_per_scale.append(pooled)
        
    return torch.cat(pooled_per_scale, dim=-1)

In [ ]:
QUESTION_PROMPT = (
    'The meme text reads: "{text}"\n'
    'Is this meme hateful, discriminatory, or offensive toward any person or group?\n'
    'Answer with "yes" or "no" only.'
)

def predict_single_image(image_path: str, text: str = "") -> tuple[float, float]:
    img = Image.open(image_path).convert('RGB')
    messages = [{'role': 'user', 'content': [
        {'type': 'image'},
        {'type': 'text', 'text': QUESTION_PROMPT.format(text=text)}
    ]}]
    
    chat_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    encoding = processor(text=chat_text, images=[img], return_tensors='pt', max_pixels=64 * 28 * 28)
    
    batch = {
        'input_ids': encoding['input_ids'].to(DEVICE),
        'attention_mask': encoding['attention_mask'].to(DEVICE),
        'pixel_values': encoding['pixel_values'].to(DEVICE, dtype=torch.float16),
        'image_grid_thw': encoding['image_grid_thw'].to(DEVICE),
    }
    if 'mm_token_type_ids' in encoding:
        batch['mm_token_type_ids'] = encoding['mm_token_type_ids'].to(DEVICE)
        
    with torch.no_grad():
        pooled = forward_multiscale(batch)
        logit = cls_head(pooled).item()
        score = 1.0 / (1.0 + np.exp(-logit))
        
    return score, logit

In [ ]:
# Chạy dự đoán cho tất cả các ảnh trong thư mục personal_images
image_extensions = {'.png', '.jpg', '.jpeg', '.webp'}
THRESHOLD = 0.5 # Ngưỡng để xác định Hateful

print("Bắt đầu dự đoán trên các ảnh cá nhân:")
has_images = False
for img_file in PERSONAL_IMAGES_DIR.iterdir():
    if img_file.suffix.lower() in image_extensions:
        has_images = True
        # Nếu ảnh có chứa text, bạn có thể truyền text tương ứng vào hàm dưới đây
        meme_text = ""
        
        try:
            score, logit = predict_single_image(str(img_file), text=meme_text)
            prediction = "Hateful" if score >= THRESHOLD else "Not Hateful"
            print(f"Ảnh: {img_file.name} | Score: {score:.4f} | Dự đoán: {prediction}")
        except Exception as e:
            print(f"Lỗi khi xử lý ảnh {img_file.name}: {e}")

if not has_images:
    print(f"Không tìm thấy ảnh nào trong thư mục {PERSONAL_IMAGES_DIR.absolute()}. Vui lòng thêm ảnh và chạy lại ô này.")